# 04 · Sentence rewrite evidence, decision offset and blending

The word and character TF-IDF baseline loses almost all of its macro-F1 deficit on the `human`/`fusion` boundary. This notebook asks four questions on the frozen holdout used by notebooks 02 and 03:

1. Do sentence-level rewrite scores, learned from training-set alignment between fusion documents and their human sources, add information that the document-level bag of n-grams misses?
2. Does a single fusion decision offset tuned on development OOF probabilities transfer to the holdout, given that related documents cross the validation boundary?
3. Does a predeclared probability average with the XLM-R head-only model help?
4. Does a fine-tuned XLM-R sentence classifier detect rewritten sentences better than character n-grams, and do its document aggregates add to the character aggregates?

Each experiment changes one component of the fixed reference and is evaluated once on the holdout with criteria fixed before evaluation. Only `train.csv` is used for fitting; no external data and no proprietary model touch the texts. This notebook restores saved results and trains nothing.

## 01 · Configuration and execution stages

All local computation runs on CPU through `scripts/run_rewrite_experiments.py`, one explicit stage at a time, with settings from `config/rewrite_experiments.toml`. Each stage writes a record with its recipe identity, output fingerprints, runtime and peak memory. The XLM-R sentence detector in section 08 is trained on a Kaggle GPU and only its sentence scores are restored here. The cell below verifies all stage fingerprints before any result is displayed.

In [1]:
import json
import sys
import tomllib
from pathlib import Path

import plotly.express as px
import plotly.io
import polars as pl
from IPython.display import display
from sklearn.metrics import roc_auc_score

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from hm3.sentence_scores import aggregate_document_scores
from hm3.validation import LABELS, fingerprint_file

plotly.io.renderers.default = "plotly_mimetype"
CONFIG = tomllib.loads((ROOT / "config/rewrite_experiments.toml").read_text())
RESULTS = ROOT / CONFIG["output_root"]
STAGES = [
    "families",
    "reference",
    "sentence_gate",
    "sentence_holdout",
    "offset",
    "blend",
    "final",
    "transformer_prepare",
    "char_all_train",
    "transformer_holdout",
    "transformer_final",
]


def load_stage(stage: str) -> dict:
    """Return the verified stage record or raise when outputs changed."""
    record = json.loads((RESULTS / stage / "record.json").read_text())
    for name, expected in record["files"].items():
        if fingerprint_file(RESULTS / stage / name) != expected:
            raise ValueError(f"Changed result file: {stage}/{name}")
    return record


records = {stage: load_stage(stage) for stage in STAGES}
display(
    pl.DataFrame(
        [
            {
                "stage": stage,
                "seconds": round(record["seconds"], 1),
                "peak_rss_gib": round(record["peak_rss_gib"], 2),
            }
            for stage, record in records.items()
        ]
    )
)
display(
    {key: CONFIG[key] for key in ["seed", "alignment", "sentence_model", "gate", "offset", "blend"]}
)

stage,seconds,peak_rss_gib
str,f64,f64
"""families""",13.2,5.0
"""reference""",1057.0,17.28
"""sentence_gate""",886.8,15.9
"""sentence_holdout""",131.0,17.63
"""offset""",284.6,2.95
…,…,…
"""final""",1394.9,27.4
"""transformer_prepare""",8.6,5.19
"""char_all_train""",1208.1,25.44


{'seed': 2026,
 'alignment': {'minimum_sentence_words': 8,
  'maximum_sentence_documents': 20,
  'minimum_shared_sentences': 2,
  'minimum_shared_share': 0.2,
  'family_folds': 3},
 'sentence_model': {'ngram_range': [2, 5],
  'min_df': 3,
  'max_features': 300000,
  'c': 1.0,
  'score_threshold': 0.5},
 'gate': {'minimum_sentence_auc': 0.9, 'minimum_document_auc': 0.8},
 'offset': {'grid_minimum': -1.0, 'grid_maximum': 1.0, 'grid_step': 0.05},
 'blend': {'xlmr_weight': 0.5}}

## 02 · Document families in the training data

**Question.** Are human and fusion documents related beyond exact duplicates, and does this affect the frozen holdout?

I split documents into sentences at sentence-final punctuation followed by whitespace and hash each sentence with at least eight words after lowercasing and collapsing whitespace. Sentences occurring in more than 20 documents are ignored as boilerplate. Two documents are linked when they share at least two such sentences and at least 20% of the smaller document's eligible sentences; families are connected components of these links. These thresholds are heuristic and were fixed before any model comparison. Exact matching counts lightly edited sentences as rewritten, so the retained share is a lower bound.

The holdout is called *family clean* for documents without any link to a development document. The official test split keeps each human text and its rewrites on the same side, so family-clean documents are the closest local analogue of test conditions.

In [2]:
assignments = pl.read_parquet(RESULTS / "families/assignments.parquet")
all_links = pl.read_parquet(RESULTS / "families/all_links.parquet")
partitions = assignments.select("id", "label", "partition")
pairs = (
    pl.concat([all_links.select("id", "id_other"), all_links.select(id="id_other", id_other="id")])
    .join(partitions, on="id")
    .join(
        partitions.rename(
            {"id": "id_other", "label": "label_other", "partition": "partition_other"}
        ),
        on="id_other",
    )
)
coverage = (
    assignments.join(
        pairs.group_by("id").agg(
            (pl.col("label_other") == "human").any().alias("has_human_relative"),
            (pl.col("label_other") == "fusion").any().alias("has_fusion_relative"),
        ),
        on="id",
        how="left",
    )
    .with_columns(pl.col("has_human_relative", "has_fusion_relative").fill_null(False))
    .group_by("label")
    .agg(
        pl.len().alias("documents"),
        pl.col("has_human_relative").mean().alias("share_with_human_relative"),
        pl.col("has_fusion_relative").mean().alias("share_with_fusion_relative"),
    )
    .sort("label")
)
crossing = (
    pairs.filter(pl.col("partition") != pl.col("partition_other"))
    .filter(pl.col("id") < pl.col("id_other"))
    .group_by("label", "label_other")
    .len()
    .sort("len", descending=True)
)
display(coverage, crossing)
display(json.loads((RESULTS / "families/summary.json").read_text()))

label,documents,share_with_human_relative,share_with_fusion_relative
str,u32,f64,f64
"""ai""",121940,0.000033,0.000016
"""fusion""",83682,0.932351,0.54694
"""human""",66730,0.02065,0.828308


label,label_other,len
str,str,u32
"""fusion""","""human""",13070
"""human""","""fusion""",12598
"""fusion""","""fusion""",9253
"""human""","""human""",336
"""ai""","""ai""",13
"""fusion""","""ai""",5
"""human""","""ai""",2
"""ai""","""human""",1


{'development_links': 70988,
 'all_links': 110447,
 'development_families': 163436,
 'purged_development_documents': 32059,
 'family_clean_holdout_documents': 30649,
 'family_fold_counts': [{'family_fold': 0, 'len': 72629},
  {'family_fold': 1, 'len': 72626},
  {'family_fold': 2, 'len': 72626}]}

In [3]:
reference = pl.read_parquet(RESULTS / "reference/holdout_probabilities.parquet")
holdout = assignments.filter(pl.col("partition") == "validation").join(
    reference.select("id", pl.col("prediction").alias("reference")), on="id", validate="1:1"
)
extent = (
    holdout.filter(pl.col("label") == "fusion")
    .with_columns(
        pl.when(pl.col("rewritten_sentences").is_null())
        .then(pl.lit("no linked human"))
        .otherwise(pl.col("rewritten_sentences").clip(upper_bound=6).cast(pl.String))
        .alias("rewritten_sentences")
    )
    .group_by("rewritten_sentences")
    .agg(
        pl.len().alias("fusion_documents"),
        (pl.col("reference") == "fusion").mean().alias("recall"),
        (pl.col("reference") == "human").mean().alias("predicted_human"),
    )
    .sort("rewritten_sentences")
)
display(extent)
display(
    px.bar(
        extent,
        x="rewritten_sentences",
        y="recall",
        text_auto=".2f",
        title="Reference fusion recall by number of rewritten eligible sentences",
        labels={
            "rewritten_sentences": "Rewritten eligible sentences (6 means six or more)",
            "recall": "Fusion recall",
        },
    )
)
relatives = (
    pairs.filter(pl.col("partition_other") == "development")
    .group_by("id")
    .agg(pl.col("label_other").unique().sort().str.join("+").alias("development_relatives"))
)
leakage = (
    holdout.join(relatives, on="id", how="left")
    .with_columns(pl.col("development_relatives").fill_null("none"))
    .filter(pl.col("label").is_in(["human", "fusion"]))
    .group_by("label", "development_relatives")
    .agg(
        pl.len().alias("documents"),
        (pl.col("reference") == pl.col("label")).mean().alias("recall"),
        (pl.col("reference") == "human").mean().alias("predicted_human"),
        (pl.col("reference") == "fusion").mean().alias("predicted_fusion"),
    )
    .sort("label", "development_relatives")
)
balance = (
    holdout.group_by("is_family_clean")
    .agg(
        pl.len().alias("documents"),
        (pl.col("label") == "human").sum().alias("true_human"),
        (pl.col("reference") == "human").sum().alias("predicted_human"),
        (pl.col("label") == "fusion").sum().alias("true_fusion"),
        (pl.col("reference") == "fusion").sum().alias("predicted_fusion"),
    )
    .sort("is_family_clean")
)
display(leakage, balance)

rewritten_sentences,fusion_documents,recall,predicted_human
str,u32,f64,f64
"""0""",690,0.36087,0.633333
"""1""",2476,0.504847,0.48021
"""2""",948,0.765823,0.223629
"""3""",1002,0.836327,0.151697
"""4""",1154,0.883016,0.112652
"""5""",1199,0.894078,0.099249
"""6""",8128,0.957554,0.032972
"""no linked human""",1141,0.805434,0.108677


label,development_relatives,documents,recall,predicted_human,predicted_fusion
str,str,u32,f64,f64,f64
"""fusion""","""fusion""",1721,0.880302,0.108077,0.880302
"""fusion""","""fusion+human""",5829,0.85778,0.135015,0.85778
"""fusion""","""human""",6717,0.797975,0.191157,0.797975
"""fusion""","""none""",2471,0.8017,0.151356,0.8017
"""human""","""ai+fusion""",2,1.0,1.0,0.0
"""human""","""fusion""",9316,0.864212,0.864212,0.134714
"""human""","""fusion+human""",212,0.948113,0.948113,0.051887
"""human""","""human""",12,0.916667,0.916667,0.083333
"""human""","""none""",3805,0.889619,0.889619,0.095401


is_family_clean,documents,true_human,predicted_human,true_fusion,predicted_fusion
bool,u32,u32,u32,u32,u32
false,23822,9542,10522,14267,13143
true,30649,3805,3832,2471,2468


**Measured observations.** 93.2% of fusion documents link to a human document and 82.8% of human documents link to a fusion document, whereas AI documents almost never link (0.003%). Human and fusion versions therefore form families, while AI texts were generated independently. Of 54,471 holdout documents, 23,822 link to a development document, and 25,668 human and fusion links cross the boundary.

Reference fusion recall falls from 0.958 with six or more rewritten eligible sentences to 0.505 with one and 0.361 with none. Relatives also shift decisions: human documents with only fusion relatives in development are predicted as fusion in 13.5% of cases versus 9.5% without relatives, and fusion documents with only human relatives reach recall 0.798 versus 0.880 with only fusion relatives. The excess of predicted human documents occurs only in the family-crossing part (10,522 predicted versus 9,542 true); the family-clean part is balanced (3,832 versus 3,805). These are associations within one split, not causal estimates.

## 03 · Reference refit with probabilities

The saved Kaggle baseline stored holdout labels but neither the development-trained model nor its probabilities. I refitted `word_char_tfidf_logistic_regression_c1` on the same 217,881 development rows with the same recipe, using the memory-bounded vocabulary construction verified in `tests/test_baseline_export.py`. Agreement with the saved Kaggle holdout labels checks that the reference is the same model.

In [4]:
reference_metrics = json.loads((RESULTS / "reference/metrics.json").read_text())
display(
    {
        key: reference_metrics[key]
        for key in [
            "macro_f1",
            "agreement_with_saved_kaggle_predictions",
            "classifier_iterations",
            "fit_vocabulary_seconds",
            "transform_seconds",
            "classifier_seconds",
            "development_matrix_nnz",
        ]
    }
)

{'macro_f1': 0.8957426929916698,
 'agreement_with_saved_kaggle_predictions': 0.9999082080373043,
 'classifier_iterations': 16,
 'fit_vocabulary_seconds': 804.1437336250001,
 'transform_seconds': 122.23585062500024,
 'classifier_seconds': 126.1327444590006,
 'development_matrix_nnz': 996837626}

**Measured observations.** The refit reproduces 99.99% of the saved Kaggle holdout labels (five of 54,471 differ), with macro-F1 0.895743 versus 0.895720. Logistic regression converged in 16 iterations. Vocabulary construction took 13.4 minutes, transforms 2.0 minutes and the classifier 2.1 minutes on the local CPU.

## 04 · Experiment 1: sentence rewrite scores

**Hypothesis.** A fusion document with one or two rewritten sentences looks almost identical to its human source in a document-level bag of n-grams. A sentence classifier trained on aligned rewritten sentences provides a local signal that the reference lacks.

**Labels.** Within development, a sentence of a fusion document is *rewritten* when its normalized hash is absent from every linked human document, and *retained* otherwise. All sentences of human documents are negatives. AI documents and fusion documents without a linked human document are unlabeled. The labels are a deterministic string comparison of training texts.

**Leakage control.** Development families linked to any holdout document are excluded from sentence-model training. Development documents receive out-of-fold scores from three family-grouped folds, so no score comes from a model trained on the same family. Holdout documents are scored by a model fitted on all remaining labeled development sentences.

**Model.** Character n-grams within word boundaries, lengths two to five, minimum document frequency 3, at most 300,000 features, sublinear TF and logistic regression with $C=1$. Seven fixed document aggregates (maximum, mean of the top two, mean, count and share of scores $\geq 0.5$, $\log(1+n)$ sentences and an empty-document indicator) are scaled to $[0, 1]$ on development with clipping (the count after $\log(1+x)$), divided by $\sqrt{7}$ and appended to the reference TF-IDF matrix. The classifier settings equal the reference.

**Gate and criteria.** The holdout comparison runs only if development OOF sentence AUC is at least 0.90 and document AUC of the maximum score for human versus fusion is at least 0.80. Success requires a holdout macro-F1 gain of at least 0.003 with a family-cluster bootstrap 95% interval above zero, no loss on the family-clean subset and a human F1 drop of at most 0.003.

In [5]:
gate = json.loads((RESULTS / "sentence_gate/gate.json").read_text())
display({key: value for key, value in gate.items() if key != "folds"}, pl.DataFrame(gate["folds"]))
sentence = json.loads((RESULTS / "sentence_holdout/metrics.json").read_text())


def summary_row(name: str, metrics: dict) -> dict:
    """Flatten aggregate and per-class F1 for one evaluated decision rule."""
    per_class = {row["label"]: row for row in metrics["holdout"]["per_class"]}
    return {
        "experiment": name,
        "holdout_macro_f1": metrics["holdout"]["macro_f1"],
        "family_clean_macro_f1": metrics["family_clean"]["macro_f1"],
        **{f"{label}_f1": per_class[label]["f1"] for label in LABELS},
        "human_precision": per_class["human"]["precision"],
        "fusion_recall": per_class["fusion"]["recall"],
    }


display(
    pl.DataFrame(
        [
            summary_row("word_char_tfidf_logistic_regression_c1", sentence["reference"]),
            summary_row("word_char_logreg_sentence_rewrite", sentence["candidate"]),
        ]
    ),
    {
        "paired_holdout": sentence["candidate"]["paired_holdout"],
        "paired_family_clean": sentence["candidate"]["paired_family_clean"],
        "success": sentence["success"],
    },
)

{'sentence_oof_auc': 0.9369696367477248,
 'document_oof_auc_max_human_vs_fusion': 0.8874419970758409,
 'training_sentences': 2072920,
 'training_documents': 84145,
 'full_fit_seconds': 248.11091691699949,
 'passed': True}

fold,fit_sentences,fit_rewritten_share,scored_sentences,fit_seconds,score_seconds
i64,i64,f64,i64,f64,f64
0,1397282,0.139362,1798059,157.77414,35.26379
1,1382030,0.140065,1790435,178.396596,33.9605
2,1366528,0.139006,1805648,155.138822,34.857002


experiment,holdout_macro_f1,family_clean_macro_f1,human_f1,ai_f1,fusion_f1,human_precision,fusion_recall
str,f64,f64,f64,f64,f64,f64,f64
"""word_char_tfidf_logistic_regre…",0.895743,0.893689,0.841125,0.989446,0.856657,0.81162,0.827817
"""word_char_logreg_sentence_rewr…",0.910541,0.908323,0.864493,0.990099,0.877032,0.835969,0.849444


{'paired_holdout': {'difference': 0.014798504786205813,
  'bootstrap_se': 0.0011804792531667043,
  'ci95_low': 0.01254707549676366,
  'ci95_high': 0.0170000763366336,
  'repetitions': 1000,
  'resampling_unit': 'all-training document family'},
 'paired_family_clean': {'difference': 0.014633799756116783},
 'success': {'minimum_gain': True,
  'positive_interval': True,
  'family_clean_not_worse': True,
  'human_f1_drop_within_limit': True,
  'passed': True}}

In [6]:
for name, metrics in [
    ("Reference", sentence["reference"]),
    ("Sentence rewrite", sentence["candidate"]),
]:
    display(
        px.imshow(
            metrics["holdout"]["row_normalized_confusion_matrix"],
            x=list(LABELS),
            y=list(LABELS),
            text_auto=".3f",
            title=f"{name}: fixed-holdout confusion proportions",
            labels={"x": "Predicted class", "y": "True class", "color": "Row proportion"},
        )
    )
recall = pl.read_parquet(RESULTS / "sentence_holdout/fusion_recall_by_rewritten_sentences.parquet")
long_recall = recall.unpivot(
    index=["rewritten_sentences_capped", "fusion_documents"],
    on=["reference_recall", "candidate_recall"],
    variable_name="model",
    value_name="recall",
)
display(
    px.bar(
        long_recall,
        x="rewritten_sentences_capped",
        y="recall",
        color="model",
        barmode="group",
        title="Fusion recall by rewritten sentences: reference versus sentence rewrite model",
        labels={
            "rewritten_sentences_capped": "Rewritten eligible sentences (4 means four or more)",
            "recall": "Fusion recall",
            "model": "Model",
        },
    )
)
display(
    pl.DataFrame(
        [{"class": label, **values} for label, values in sentence["aggregate_coefficients"].items()]
    )
)

class,max_score,top2_mean_score,mean_score,count_over_threshold,share_over_threshold,log_sentences,has_no_sentences
str,f64,f64,f64,f64,f64,f64,f64
"""ai""",-3.311848,-0.88204,13.748035,5.801154,0.534394,3.637102,1.68424
"""fusion""",1.788864,8.818104,-4.65355,-0.152312,2.950958,-4.623446,-0.85407
"""human""",1.52157,-7.946062,-9.095658,-5.633674,-3.482974,0.980387,-0.83017


**Measured observations.** The gate passed with OOF sentence AUC 0.937 and document AUC 0.887, using 2,072,920 labeled sentences from 84,145 development documents. Holdout macro-F1 rises from 0.895743 to 0.910541, a paired gain of 0.0148 with family-cluster bootstrap 95% interval $[0.0125, 0.0170]$. The family-clean gain is 0.0146 (0.8937 to 0.9083). Human F1 rises from 0.841 to 0.864 and fusion F1 from 0.857 to 0.877, and both error directions decrease: human to fusion from 1,630 to 1,348 and fusion to human from 2,631 to 2,276. Fusion recall improves for every group with at least one rewritten sentence (one: 0.505 to 0.536; two: 0.766 to 0.822; three: 0.836 to 0.903) and falls for documents without a rewritten eligible sentence (0.361 to 0.245), which the sentence model cannot separate from human text. All predeclared criteria are met.

**Implementation note.** The first joint fit used standardized aggregates. I stopped it before it produced any metric: the rare empty-document indicator reached $z = 49.5$, so the maximum squared row norm rose from 2 to 2,475 and saga's step size shrank by about three orders of magnitude. The reported run uses the bounded scaling above, which keeps the maximum squared row norm below 2.8; the classifier then converges in 16 iterations like the reference.

The aggregate coefficients put positive fusion weight on the top-two mean and the share above the threshold, and negative human weight on the mean and top-two mean, which matches localized rewriting. The aggregates are correlated, so individual coefficients should not be interpreted in isolation.

## 05 · Experiment 2: fusion decision offset

**Hypothesis.** The reference predicts `human` more often than the true count only in the family-crossing part of the holdout. An offset tuned on leakage-prone folds should therefore favour fusion and fail to help family-clean documents, whereas an offset tuned on family-grouped folds should stay near zero.

I add a constant $\delta$ to $\log P(\text{fusion})$ and predict the class with the highest adjusted score. $\delta$ is selected from $[-1, 1]$ in steps of 0.05 by OOF macro-F1 on the frozen 30,002-document development CV sample, with ties broken by the smallest $|\delta|$. Two fold schemes use identical documents and recipes: the frozen exact-text CV folds and new family-grouped folds. Each selected $\delta$ is applied unchanged to the reference holdout probabilities. A nonzero $\delta$ from the family-grouped scheme is adopted only if it does not reduce macro-F1 on either the full holdout or the family-clean subset. The CV models are trained on about 20,000 rows, so their calibration can differ from the 217,881-row reference.

In [7]:
offset = json.loads((RESULTS / "offset/metrics.json").read_text())
curves = pl.read_parquet(RESULTS / "offset/oof_offset_curves.parquet")
display(
    px.line(
        curves,
        x="offset",
        y="macro_f1",
        color="scheme",
        title="Development OOF macro-F1 by fusion log-probability offset",
        labels={
            "offset": "Fusion log-probability offset",
            "macro_f1": "OOF macro-F1",
            "scheme": "Fold scheme",
        },
    )
)
display(
    pl.DataFrame(
        [
            {
                "scheme": scheme,
                "selected_offset": values["selected_offset"],
                "oof_macro_f1_at_zero": values["oof_macro_f1_at_zero"],
                "oof_macro_f1_selected": values["oof_macro_f1_selected"],
                "holdout_difference": values["paired_holdout"]["difference"],
                "holdout_ci95_low": values["paired_holdout"]["ci95_low"],
                "holdout_ci95_high": values["paired_holdout"]["ci95_high"],
                "family_clean_difference": values["paired_family_clean"]["difference"],
            }
            for scheme, values in offset["schemes"].items()
        ]
    ),
    {
        "adoption": offset["adoption"],
        "frozen_fold_label_reproduction": offset["frozen_fold_label_reproduction"],
    },
)

scheme,selected_offset,oof_macro_f1_at_zero,oof_macro_f1_selected,holdout_difference,holdout_ci95_low,holdout_ci95_high,family_clean_difference
str,f64,f64,f64,f64,f64,f64,f64
"""frozen_cv_folds""",-0.05,0.847978,0.848542,0.0003,-0.000357,0.000985,0.002424
"""family_grouped_folds""",-0.05,0.851458,0.852015,0.0003,-0.000357,0.000985,0.002424


{'adoption': {'offset': -0.05,
  'holdout_not_worse': True,
  'family_clean_not_worse': True,
  'nonzero_offset': True,
  'adopted': True},
 'frozen_fold_label_reproduction': 0.9994333711085928}

**Measured observations.** Both fold schemes select $\delta = -0.05$, a negligible shift toward human, so the hypothesis that leakage-prone folds would favour fusion is not supported. On the holdout the paired difference is 0.0003 with 95% interval $[-0.0004, 0.0010]$; the family-clean difference is 0.0024. The predeclared rule formally adopts the offset because neither metric decreases, but the measured effect is indistinguishable from zero: argmax is already close to the macro-F1 optimum for this model. Family-grouped folds give higher OOF macro-F1 than the frozen exact-text folds (0.8515 versus 0.8480 at $\delta = 0$), so related documents across folds make this CV pessimistic rather than optimistic. Frozen-fold refits reproduce 99.94% of the saved OOF labels.

## 06 · Fallback: TF-IDF and XLM-R probability average

The weight is fixed at 0.5 before evaluation because no development OOF probabilities exist for both models. The XLM-R model is the saved head-only 40k run with its known gradient-accumulation loss scaling, so this measures the complementarity of the existing encoder rather than of a tuned one. The blend is adopted only if it improves holdout macro-F1 without a family-clean loss.

In [8]:
blend = json.loads((RESULTS / "blend/metrics.json").read_text())
display(
    pl.DataFrame([summary_row("word_char_logreg_xlmr_blend", blend)]),
    {
        "paired_holdout": blend["paired_holdout"],
        "paired_family_clean": blend["paired_family_clean"],
        "adopted": blend["adopted"],
    },
)

experiment,holdout_macro_f1,family_clean_macro_f1,human_f1,ai_f1,fusion_f1,human_precision,fusion_recall
str,f64,f64,f64,f64,f64,f64,f64
"""word_char_logreg_xlmr_blend""",0.888828,0.893095,0.84036,0.98583,0.840294,0.786606,0.778946


{'paired_holdout': {'difference': -0.006914763858922979,
  'bootstrap_se': 0.0013028052398467824,
  'ci95_low': -0.009372875043245552,
  'ci95_high': -0.004196216768003736,
  'repetitions': 1000,
  'resampling_unit': 'all-training document family'},
 'paired_family_clean': {'difference': -0.0005937132855028393},
 'adopted': False}

**Measured observations.** The equal-weight blend lowers holdout macro-F1 to 0.888828, a paired difference of $-0.0069$ with 95% interval $[-0.0094, -0.0042]$; the family-clean difference is $-0.0006$. Fusion recall drops from 0.828 to 0.779 and human precision from 0.812 to 0.787. The blend is not adopted: equal weighting gives too much influence to an encoder that is weaker on the hard fusion documents.

## 07 · Comparison and exported submissions

All rows use the same 54,471 holdout documents and the reference class order human, ai, fusion. Differences are paired against the reference. Submission files are exported only for adopted decisions and validated against `sample_submission.csv`; none is submitted automatically.

In [9]:
comparison = pl.DataFrame(
    [
        {
            "experiment": "word_char_tfidf_logistic_regression_c1",
            "changed_setting": "reference refit with probabilities",
            "validation": "fixed holdout, 217,881 development rows",
            "holdout_macro_f1": sentence["reference"]["holdout"]["macro_f1"],
            "family_clean_macro_f1": sentence["reference"]["family_clean"]["macro_f1"],
            "difference": 0.0,
        },
        {
            "experiment": "word_char_logreg_sentence_rewrite",
            "changed_setting": "plus seven sentence rewrite aggregates",
            "validation": "same holdout; family-purged sentence training",
            "holdout_macro_f1": sentence["candidate"]["holdout"]["macro_f1"],
            "family_clean_macro_f1": sentence["candidate"]["family_clean"]["macro_f1"],
            "difference": sentence["candidate"]["paired_holdout"]["difference"],
        },
        *[
            {
                "experiment": f"word_char_logreg_fusion_offset_{scheme}",
                "changed_setting": f"fusion offset {values['selected_offset']:+.2f}",
                "validation": "offset tuned on 30k development OOF; same holdout",
                "holdout_macro_f1": values["holdout"]["macro_f1"],
                "family_clean_macro_f1": values["family_clean"]["macro_f1"],
                "difference": values["paired_holdout"]["difference"],
            }
            for scheme, values in offset["schemes"].items()
        ],
        {
            "experiment": "word_char_logreg_xlmr_blend",
            "changed_setting": "equal-weight probability average with XLM-R head-only",
            "validation": "predeclared weight; same holdout",
            "holdout_macro_f1": blend["holdout"]["macro_f1"],
            "family_clean_macro_f1": blend["family_clean"]["macro_f1"],
            "difference": blend["paired_holdout"]["difference"],
        },
    ]
)
display(comparison)
display(json.loads((RESULTS / "final/exports.json").read_text()))

experiment,changed_setting,validation,holdout_macro_f1,family_clean_macro_f1,difference
str,str,str,f64,f64,f64
"""word_char_tfidf_logistic_regre…","""reference refit with probabili…","""fixed holdout, 217,881 develop…",0.895743,0.893689,0.0
"""word_char_logreg_sentence_rewr…","""plus seven sentence rewrite ag…","""same holdout; family-purged se…",0.910541,0.908323,0.014799
"""word_char_logreg_fusion_offset…","""fusion offset -0.05""","""offset tuned on 30k developmen…",0.896042,0.896113,0.0003
"""word_char_logreg_fusion_offset…","""fusion offset -0.05""","""offset tuned on 30k developmen…",0.896042,0.896113,0.0003
"""word_char_logreg_xlmr_blend""","""equal-weight probability avera…","""predeclared weight; same holdo…",0.888828,0.893095,-0.006915


{'baseline_test_reproduction': 1.0,
 'word_char_logreg_fusion_offset_all_train': {'path': 'runs/rewrite_experiments/final/submissions/word_char_logreg_fusion_offset_all_train.csv',
  'predicted_share': {'human': 0.27272, 'ai': 0.5182, 'fusion': 0.20908}},
 'word_char_logreg_sentence_rewrite_all_train': {'path': 'runs/rewrite_experiments/final/submissions/word_char_logreg_sentence_rewrite_all_train.csv',
  'predicted_share': {'human': 0.27, 'ai': 0.51712, 'fusion': 0.21288}},
 'seconds': 1394.2615119580005}

## 08 · Experiment 3: XLM-R sentence detector

**Hypothesis.** The character n-gram sentence classifier misses rewrites that keep the surface character patterns of their source. A pretrained multilingual encoder fine-tuned on the same rewrite labels should rank rewritten sentences better, and its document aggregates should add information that the character aggregates lack.

**Design.** I fine-tune `FacebookAI/xlm-roberta-base` (revision `e73636d`) for one epoch as a binary rewritten versus retained sentence classifier: at most 96 tokens per sentence, learning rate $2 \times 10^{-5}$ with 5% linear warmup and linear decay, weight decay 0.01, batch size 64, FP16 and seed 2026 on two Kaggle T4 GPUs. The training set is the 675,638 labeled sentences of development family fold 0 (27,611 documents, 14.0% rewritten), without families linked to the holdout. 20,000 labeled sentences from the other folds monitor sentence AUC during training. The fitted encoder scores all 7,332,992 eligible train and test sentences, and the seven aggregates and bounded scaling of section 04 are computed from these scores. The private Kaggle bundle holds sentence keys, labels and counts but no text; the hosted notebook reads texts from the attached competition data and verifies that its sentence split reproduces the local counts.

**Leakage control.** The encoder has seen fold 0, so all 27,622 development documents of those families are removed from every logistic regression fit in this section, and sample weights restore the original development class totals. Three classifiers share the remaining 190,259 rows and the reference TF-IDF matrix: character aggregates only (control), character plus XLM-R aggregates (candidate) and XLM-R aggregates only (ablation). The control differs from the experiment 1 candidate only by the removed rows, so the comparison isolates the encoder.

**Criteria.** Fixed before evaluation and identical to experiment 1: a candidate versus control gain of at least 0.003 with a family-cluster bootstrap 95% interval above zero, no loss on the family-clean subset and a human F1 drop of at most 0.003. The submission refits the candidate on all labeled training rows except the families of the encoder training documents.

In [10]:
transformer = json.loads((RESULTS / "transformer_holdout/metrics.json").read_text())
score_record = json.loads(
    (ROOT / "runs/kaggle/hw3-xlmr-sentence-rewrite/sentence_scores.json").read_text()
)
display(
    pl.DataFrame(score_record["history"]),
    {
        key: score_record[key]
        for key in [
            "model_id",
            "revision",
            "training_seconds",
            "scoring_seconds",
            "scored_sentences",
        ]
    },
    score_record["environment"],
    transformer["gate"],
)
# Document AUC of the maximum sentence score on the same kept development rows.
kept_pairs = (
    pl.read_parquet(RESULTS / "reference/development_rows.parquet")
    .select("id", "label")
    .join(
        pl.read_parquet(RESULTS / "transformer_prepare/excluded_development_documents.parquet"),
        on="id",
        how="anti",
    )
    .filter(pl.col("label").is_in(["human", "fusion"]))
)
sentence_scores = pl.read_parquet(
    ROOT / "runs/kaggle/hw3-xlmr-sentence-rewrite/sentence_scores.parquet"
)
document_maxima = {
    "character n-grams": pl.read_parquet(RESULTS / "sentence_gate/development_aggregates.parquet"),
    "XLM-R": aggregate_document_scores(
        sentence_scores.join(kept_pairs.select("id"), on="id"),
        kept_pairs["id"],
        CONFIG["sentence_model"]["score_threshold"],
    ),
}
display(
    pl.DataFrame(
        [
            {
                "sentence_detector": name,
                "documents": kept_pairs.height,
                "document_auc_max_human_vs_fusion": roc_auc_score(
                    joined["label"] == "fusion", joined["max_score"]
                ),
            }
            for name, aggregates in document_maxima.items()
            for joined in [kept_pairs.join(aggregates, on="id", how="left", maintain_order="left")]
        ]
    )
)
display(
    pl.DataFrame(
        [
            summary_row(name, transformer["evaluations"][name])
            for name in ["control_char", "candidate_char_transformer", "ablation_transformer"]
        ]
    ),
    pl.DataFrame(
        [
            {
                "comparison": comparison,
                "holdout_difference": transformer["evaluations"][name]["paired_holdout"][
                    "difference"
                ],
                "ci95_low": transformer["evaluations"][name]["paired_holdout"]["ci95_low"],
                "ci95_high": transformer["evaluations"][name]["paired_holdout"]["ci95_high"],
                "family_clean_difference": transformer["evaluations"][name]["paired_family_clean"][
                    "difference"
                ],
            }
            for name, comparison in [
                ("candidate_char_transformer", "character plus XLM-R versus character only"),
                ("ablation_transformer", "XLM-R only versus character only"),
                (
                    "candidate_versus_previous",
                    "character plus XLM-R versus submitted sentence rewrite",
                ),
            ]
        ]
    ),
    {"success": transformer["success"], "fits": transformer["fits"]},
)
display(
    px.imshow(
        transformer["evaluations"]["candidate_char_transformer"]["holdout"][
            "row_normalized_confusion_matrix"
        ],
        x=list(LABELS),
        y=list(LABELS),
        text_auto=".3f",
        title="Character plus XLM-R sentence aggregates: fixed-holdout confusion proportions",
        labels={"x": "Predicted class", "y": "True class", "color": "Row proportion"},
    )
)
display(json.loads((RESULTS / "transformer_final/exports.json").read_text()))

step,monitor_auc,seconds
i64,f64,f64
2639,0.970241,1214.468325
5278,0.976608,2425.951193
7917,0.979517,3639.030517
10557,0.979999,4850.317061


{'model_id': 'FacebookAI/xlm-roberta-base',
 'revision': 'e73636d4f797dec63c3081bb6ed5c7b0bb3f2089',
 'training_seconds': 4850.319598027,
 'scoring_seconds': 5314.863217196999,
 'scored_sentences': 7332992}

{'python': '3.13.16',
 'torch': '2.11.0+cu130',
 'cuda': '13.0',
 'gpus': ['Tesla T4', 'Tesla T4'],
 'polars': '2.1.0',
 'transformers': '5.24.0',
 'tokenizers': '0.23.3',
 'scikit-learn': '1.6.1',
 'free_disk_gib': 19.5}

{'monitor_sentence_auc': 0.9799991163676958,
 'document_auc_max_human_vs_fusion': 0.9243573359031769,
 'kept_development_rows': 190259,
 'excluded_development_rows': 27622}

sentence_detector,documents,document_auc_max_human_vs_fusion
str,i64,f64
"""character n-grams""",92706,0.88604
"""XLM-R""",92706,0.924357


experiment,holdout_macro_f1,family_clean_macro_f1,human_f1,ai_f1,fusion_f1,human_precision,fusion_recall
str,f64,f64,f64,f64,f64,f64,f64
"""control_char""",0.906936,0.903766,0.858171,0.99002,0.872618,0.833392,0.848011
"""candidate_char_transformer""",0.923584,0.914119,0.884088,0.990771,0.895892,0.863169,0.875553
"""ablation_transformer""",0.920992,0.913666,0.879627,0.99042,0.89293,0.863348,0.877166


comparison,holdout_difference,ci95_low,ci95_high,family_clean_difference
str,f64,f64,f64,f64
"""character plus XLM-R versus ch…",0.016648,0.014692,0.018602,0.010353
"""XLM-R only versus character on…",0.014056,0.01173,0.01643,0.0099
"""character plus XLM-R versus su…",0.013043,0.011143,0.014912,0.005797


{'success': {'minimum_gain': True,
  'positive_interval': True,
  'family_clean_not_worse': True,
  'human_f1_drop_within_limit': True,
  'passed': True},
 'fits': {'control_char': {'classifier_seconds': 106.6819863339988,
   'classifier_iterations': 17},
  'candidate_char_transformer': {'classifier_seconds': 109.10425208300148,
   'classifier_iterations': 15},
  'ablation_transformer': {'classifier_seconds': 98.11216033299934,
   'classifier_iterations': 15}}}

{'word_char_logreg_xlmr_sentence_rewrite_all_train': {'path': 'runs/rewrite_experiments/transformer_final/submissions/word_char_logreg_xlmr_sentence_rewrite_all_train.csv',
  'predicted_share': {'human': 0.26324, 'ai': 0.5152, 'fusion': 0.22156}}}

**Measured observations.** Monitor sentence AUC rises from 0.970 after a quarter of the epoch to 0.980 at its end; training took 81 minutes and scoring 89 minutes. On the same 92,706 kept development human and fusion documents, the document AUC of the maximum sentence score is 0.924 for XLM-R versus 0.886 for character n-grams.

Holdout macro-F1 is 0.906936 for the control, 0.923584 for the candidate and 0.920992 for the ablation. The candidate gain over the control is 0.0166 with 95% interval $[0.0147, 0.0186]$ and 0.0104 on the family-clean subset (0.9038 to 0.9141). Human F1 rises from 0.858 to 0.884 and fusion F1 from 0.873 to 0.896; human to fusion errors fall from 1,479 to 1,210 and fusion to human errors from 2,300 to 1,853. All predeclared criteria are met. The XLM-R aggregates alone exceed the control by 0.0141, and adding the character aggregates raises macro-F1 by a further 0.0026 (not tested separately), so the encoder carries most of the sentence signal.

Removing the fold 0 families costs the control 0.0036 relative to the experiment 1 candidate trained on all development rows (0.910541). The candidate still exceeds that model by 0.0130 with 95% interval $[0.0111, 0.0149]$ and by 0.0058 on the family-clean subset.

## 09 · Conclusions and limitations

Sentence rewrite evidence is the only direction with clear gains. Character sentence aggregates add 0.0148 holdout macro-F1 to the reference, and XLM-R sentence aggregates add a further 0.0166 over a control fitted on the same rows; both intervals exclude zero and both gains hold on family-clean documents. The fusion offset is within noise, and the equal-weight blend with the document-level XLM-R is rejected.

`word_char_logreg_xlmr_sentence_rewrite_all_train.csv` is fitted on 244,730 training rows (all 272,352 labeled rows except the 27,622 in families of the encoder training documents) with all-training family folds for out-of-fold character sentence scores, encoder scores for every training and test sentence and the saved all-training TF-IDF matrix. 96.0% of its test labels agree with `word_char_logreg_sentence_rewrite_all_train.csv`, and its predicted fusion share is 22.2%. The earlier file uses all labeled rows; 95.7% of its labels agree with the baseline submission. The offset file exists because its adoption rule was met, but its effect is within noise.

**Limitations.** Family links use heuristic thresholds. Exact sentence matching labels lightly edited sentences as rewritten, and rewriting that touches only short sentences remains undetected. I formed the rewrite hypothesis partly from holdout error analysis; the same pattern appeared independently in development CV OOF predictions, and every holdout comparison was evaluated once with fixed criteria. The encoder is trained on one family fold for one epoch, and GPU training is not bit-for-bit deterministic. The rows it saw are dropped from the logistic regression, which costs the control about 0.004 macro-F1. Results use one seed, and the family-cluster bootstrap does not cover seed variance. Kaggle public scores are not used for any selection.